In [1]:
import torch
import torch.nn as nn
import glob
import os
import time
import pickle
import inspect
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from pathlib import Path

from utils import functions as uf
from utils.model import DynamicMLP

In [2]:
folder_path = './data/'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') #without cuda it was 21.6 now its 14.2


csv_files = glob.glob(os.path.join(folder_path, '*c000.csv'))
df_all = pd.concat((pd.read_csv(file, sep=";") for file in csv_files), ignore_index=True)

forget_df = pd.read_csv(os.path.join(folder_path, 'forget_data.csv'), sep=",")

random_seed = 42
id_col = "user_id"

#train/validation split for fine tuning

forget_ids = set(forget_df[id_col])
clean_df = df_all[~df_all[id_col].isin(forget_ids)].reset_index(drop=True)

ids = clean_df[id_col].unique()
rng = np.random.default_rng(random_seed)
rng.shuffle(ids)
val_frac = 0.15
n_val = int(len(ids) * val_frac)
val_ids = set(ids[:n_val])

val_df = clean_df[clean_df[id_col].isin(val_ids)].reset_index(drop=True)
train_df = clean_df[~clean_df[id_col].isin(val_ids)].reset_index(drop=True)


X_train, y_train, feature_cols, target_cols = uf.prepare_data(train_df, id_col=id_col, target_prefix='target__')
X_val, y_val, _, _ = uf.prepare_data(val_df, id_col=id_col, target_prefix='target__')

imputer = SimpleImputer(strategy='median')
X_train = imputer.fit_transform(X_train).astype(np.float32)
X_val = imputer.transform(X_val).astype(np.float32)

DataFrame columns: ['ang_m_datediff_att_linea', 'ang_m_flag_mnp', 'ang_m_datediff_cessazione', 'ang_m_datediff_scad_sim', 'ang_m_cns_cre_res_per', 'ang_m_anz_linea', 'ang_m_datediff_mnp_in', 'ang_m_datediff_cam_prof', 'ang_m_datediff_mnp_out', 'ohe_ang_m_stato_linea_a', 'ohe_ang_m_stato_linea_c', 'ohe_ang_m_stato_linea_n', 'ohe_ang_m_stato_linea_s', 'ohe_ang_m_tipo_linea_fwa', 'ohe_ang_m_tipo_linea_pp', 'ohe_ang_m_tipo_linea_xt', 'ohe_ang_m_area_territoriale_altro_nd', 'ohe_ang_m_area_territoriale_c', 'ohe_ang_m_area_territoriale_ne', 'ohe_ang_m_area_territoriale_no', 'ohe_ang_m_area_territoriale_s', 'ohe_ang_m_pdv_canale_des_top_4_altro', 'ohe_ang_m_pdv_canale_des_top_4_grande_distribuzione', 'ohe_ang_m_pdv_canale_des_top_4_null', 'ohe_ang_m_pdv_canale_des_top_4_one_team', 'ohe_ang_m_pdv_canale_des_top_4_retail', 'ohe_ang_m_pdv_canale_des_top_4_tele_sales', 'ohe_ang_m_mnp_olo_prov_infrastrutturati', 'ohe_ang_m_mnp_olo_prov_mvno', 'ohe_ang_m_mnp_olo_prov_nd', 'ohe_ang_m_mnp_olo_dest_in

In [3]:
val_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 18104 entries, 0 to 18103
Columns: 410 entries, ang_m_datediff_att_linea to user_id
dtypes: float64(110), int64(300)
memory usage: 56.6 MB


In [4]:
val_df.head()

,ang_m_datediff_att_linea,ang_m_flag_mnp,ang_m_datediff_cessazione,ang_m_datediff_scad_sim,ang_m_cns_cre_res_per,ang_m_anz_linea,ang_m_datediff_mnp_in,ang_m_datediff_cam_prof,ang_m_datediff_mnp_out,ohe_ang_m_stato_linea_a,...,target__family_and_parenting,target__sports,target__food_and_drink,target__pets,target__performance,target__business,target__campagne,target__roaming,target__assicurazioni,user_id
0,8,1,-1,-10,5.000000,7,7,-1,-1,1,...,0,0,1,0,1,1,1,0,0,2192
1,22,1,-1,-11,4.580000,22,22,-1,-1,1,...,0,0,1,1,0,0,0,0,0,3676
2,60,0,-1,-10,9.536857,60,-1,-1,-1,1,...,0,0,0,0,0,0,0,0,0,4464
3,20,1,18,-9,-3.758361,20,20,-1,18,0,...,0,0,0,0,0,0,0,0,0,5661
4,112,1,-1,-11,12.162089,111,112,72,-1,1,...,0,0,0,0,0,0,0,0,0,6537
